# 04 Dataset modelowy dla `event_threshold_10`

Celem notebooka jest przygotowanie roboczego datasetu predykcyjnego dla pierwszych modeli. Nie trenujemy tu jeszcze modeli; sprawdzamy, czy target, cechy i splity są metodologicznie gotowe do notebooka 05.

Decyzje przeniesione z notebooka 03:

- główny indeks: `BWCI_pu`,
- okno bazowe: 10 obserwowanych rekordów,
- główny target klasyfikacyjny: `event_threshold_10`,
- pomocniczy target regresyjny: `BWCI_future_10`,
- `event_drop_10` zostaje na późniejszą analizę wrażliwości.

Najważniejsza zasada: cechy muszą być dostępne w czasie `t`, a target pochodzić z przyszłości `t + 10`.


In [1]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT_CANDIDATES: list[Path] = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "AGENTS.md").exists()), Path.cwd())

DATA_DIR_CANDIDATES: list[Path] = [
  PROJECT_ROOT / "data/BEHACOM/Behacom",
  Path("data/BEHACOM/Behacom"),
  Path("../data/BEHACOM/Behacom"),
]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), DATA_DIR_CANDIDATES[0])
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

ENCODING = "latin-1"
EXCLUDE_USERS: list[int] = [2]
WINDOW = 10
HORIZON = 10
TARGET_COL = "event_threshold_10"
REGRESSION_TARGET_COL = "BWCI_future_10"
OUTPUT_DATASET_PATH = OUTPUT_DIR / "model_dataset_event_threshold_10_v0_1.parquet"
OUTPUT_METADATA_PATH = OUTPUT_DIR / "model_dataset_event_threshold_10_v0_1_metadata.json"

sns.set_theme(style="whitegrid", palette="muted", font_scale=0.9)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")


## Sekcja 1: Odtworzenie BWCI v0.1 dla okna 10

Notebook 04 jest samowystarczalny: odtwarza dokładnie roboczą definicję BWCI v0.1 potrzebną do datasetu modelowego. Na tym etapie nie przenosimy kodu do `src/bwci/`, bo definicja indeksu może jeszcze ulec zmianie po pierwszych modelach i analizie wrażliwości.

Ładujemy tylko kolumny potrzebne do konstrukcji indeksu i przyszłych cech modelowych. Nadal pomijamy digraphy, per-key columns i problematyczne click-speed columns.


In [2]:
def discover_csv_files(data_dir: Path) -> list[Path]:
  """ Discover BEHACOM user CSV files sorted by numeric user id.

    Args:
      data_dir: BEHACOM directory containing User* subdirectories.

    Returns:
      list[Path]: CSV file paths sorted by numeric user id.
  """

  def sort_key(path: Path) -> int:
    """ Extract a numeric user id from a BEHACOM CSV path. """
    match = re.search(r"User(\d+)", path.parent.name)
    return int(match.group(1)) if match else 10_000

  return sorted(data_dir.glob("User*/User*_BEHACOM.csv"), key=sort_key)


def get_bwci_source_columns(sample_path: Path) -> list[str]:
  """ Select source columns used by the BWCI v0.1 model dataset.

    Args:
      sample_path: One BEHACOM CSV used to inspect the header.

    Returns:
      list[str]: Column names to load.
  """
  header = pd.read_csv(sample_path, encoding=ENCODING, nrows=0).columns.tolist()
  skip_prefixes = (
    "digraph_",
    "keystrokes_key_",
    "press_release_average_",
    "click_speed_average_",
    "click_speed_stddev_",
  )
  keep: list[str] = []
  for col in header:
    if col == "press_release_average_interval":
      keep.append(col)
      continue
    if any(col.startswith(prefix) for prefix in skip_prefixes):
      continue
    keep.append(col)
  return keep


def load_behacom_core(data_dir: Path) -> pd.DataFrame:
  """ Load selected BEHACOM columns and add helper variables.

    Args:
      data_dir: BEHACOM directory containing user CSV files.

    Returns:
      pd.DataFrame: Combined selected dataset with helper columns.
  """
  files = discover_csv_files(data_dir)
  if not files:
    raise FileNotFoundError(f"No BEHACOM CSV files found under {data_dir.resolve()}")

  columns = get_bwci_source_columns(files[0])
  frames: list[pd.DataFrame] = []
  for path in files:
    chunk = pd.read_csv(path, encoding=ENCODING, usecols=columns, low_memory=False)
    frames.append(chunk)

  out = pd.concat(frames, ignore_index=True)
  out["USER"] = out["USER"].astype(int)
  out = out[~out["USER"].isin(EXCLUDE_USERS)].copy()
  out["datetime"] = pd.to_datetime(out["timestamp"], unit="ms", errors="coerce")
  out = out.sort_values(["USER", "datetime"]).reset_index(drop=True)
  out["obs_idx"] = out.groupby("USER").cumcount()

  mouse_action_cols = [col for col in out.columns if col.startswith("mouse_action_counter_")]
  out["mouse_total"] = out[mouse_action_cols].sum(axis=1)
  out["is_idle"] = (out["keystroke_counter"] == 0) & (out["mouse_total"] == 0)
  out["foreground_time_was_capped"] = out["current_app_foreground_time"] > 60
  out["current_app_foreground_time"] = out["current_app_foreground_time"].clip(upper=60)
  return out


df = load_behacom_core(DATA_DIR)
print(f"Loaded: {len(df):,} rows, {df['USER'].nunique()} users, {len(df.columns)} columns")
print(f"Excluded users: {EXCLUDE_USERS}")
print(f"Foreground time capped rows: {df['foreground_time_was_capped'].sum():,}")

assert 2 not in df["USER"].unique()
assert df["datetime"].notna().all()


Loaded: 166,879 rows, 11 users, 82 columns
Excluded users: [2]
Foreground time capped rows: 55


### Interpretacja ładowania

W tym notebooku nie wykonujemy pełnego audytu danych od nowa. Korzystamy z decyzji z poprzednich etapów: User 2 pozostaje wykluczony, a wartości `current_app_foreground_time > 60` są traktowane jako artefakt i capowane do jednej minuty.

Dodajemy `obs_idx`, czyli kolejny numer obserwacji per użytkownik. To będzie stabilniejszy klucz czasowy dla przesunięć i splitów niż sam timestamp, ponieważ w BEHACOM nie zakładamy idealnej, unikalnej siatki minutowej.


In [3]:
def build_rolling_features(source_df: pd.DataFrame, window: int) -> pd.DataFrame:
  """ Build rolling features for BWCI construction.

    Args:
      source_df: Cleaned 1-minute BEHACOM data sorted by user and time.
      window: Rolling window size in observed rows.

    Returns:
      pd.DataFrame: Rolling features with one row per source row after filtering fully idle windows.
  """
  results: list[pd.DataFrame] = []
  min_periods = max(1, window // 2)

  for uid in sorted(source_df["USER"].unique()):
    udf = source_df[source_df["USER"] == uid].reset_index(drop=True)

    rolled = pd.DataFrame(
      {
        "USER": uid,
        "obs_idx": udf["obs_idx"],
        "datetime": udf["datetime"],
        "keystroke_sum": udf["keystroke_counter"].rolling(window, min_periods=min_periods).sum(),
        "mouse_sum": udf["mouse_total"].rolling(window, min_periods=min_periods).sum(),
        "word_sum": udf["word_counter"].rolling(window, min_periods=min_periods).sum(),
        "erase_sum": udf["erase_keys_counter"].rolling(window, min_periods=min_periods).sum(),
        "switch_sum": udf["changes_between_apps"].rolling(window, min_periods=min_periods).sum(),
        "idle_ratio": udf["is_idle"].rolling(window, min_periods=min_periods).mean(),
        "press_press_mean": udf["press_press_average_interval"].rolling(window, min_periods=min_periods).mean(),
        "press_press_std": udf["press_press_stddev_interval"].rolling(window, min_periods=min_periods).mean(),
        "foreground_time_mean": udf["current_app_foreground_time"].rolling(window, min_periods=min_periods).mean(),
        "foreground_time_capped_sum": udf["foreground_time_was_capped"].rolling(window, min_periods=min_periods).sum(),
        "active_apps_mean": udf["active_apps_average"].rolling(window, min_periods=min_periods).mean(),
      }
    )
    rolled["window"] = window
    results.append(rolled)

  out = pd.concat(results, ignore_index=True)
  out = out[out["idle_ratio"] < 1.0].copy()
  out["input_sum"] = out["keystroke_sum"] + out["mouse_sum"]
  out["erase_ratio"] = np.where(out["keystroke_sum"] > 0, out["erase_sum"] / out["keystroke_sum"], np.nan)
  out["typing_cv"] = np.where(out["press_press_mean"] > 0, out["press_press_std"] / out["press_press_mean"], np.nan)
  out["foreground_focus_share"] = (out["foreground_time_mean"] / 60).clip(0, 1)
  return out


rolling10 = build_rolling_features(df, WINDOW)
rolling10_summary = pd.DataFrame(
  [
    {
      "window": WINDOW,
      "rows": len(rolling10),
      "users": rolling10["USER"].nunique(),
      "median_idle_ratio": rolling10["idle_ratio"].median(),
      "median_input_sum": rolling10["input_sum"].median(),
      "median_switch_sum": rolling10["switch_sum"].median(),
      "median_foreground_focus_share": rolling10["foreground_focus_share"].median(),
    }
  ]
)
rolling10_summary


,window,rows,users,median_idle_ratio,median_input_sum,median_switch_sum,median_foreground_focus_share
0,10,76149,11,0.2000,6157.0000,1.0000,0.9439


### Interpretacja rolling features

Okno 10 traktujemy jako roboczy kompromis: jest wystarczająco krótkie, żeby opisywać lokalny stan pracy, ale zawiera więcej informacji niż pojedyncza minuta. Wiersze w pełni idle są odfiltrowane, ponieważ BWCI ma opisywać ciągłość obserwowalnej pracy, a nie okresy całkowitego braku interakcji.

W tym miejscu powstają cechy dostępne w czasie `t`. Target zostanie dodany dopiero przez przesunięcie przyszłego BWCI.


In [4]:
def robust_percentile_by_user(df: pd.DataFrame, col: str) -> pd.Series:
  """ Compute per-user percentile ranks for a column.

    Args:
      df: DataFrame containing USER and the target column.
      col: Column to rank.

    Returns:
      pd.Series: Percentile ranks in [0, 1].
  """
  return df.groupby("USER")[col].rank(pct=True)


def inverse_robust_scale_by_user(df: pd.DataFrame, col: str, q: float = 0.95) -> pd.Series:
  """ Convert an adverse metric into a per-user [0, 1] stability score.

    Args:
      df: DataFrame containing USER and the target column.
      col: Column where larger values are worse.
      q: Upper quantile used as robust cap.

    Returns:
      pd.Series: Score where larger values indicate better continuity.
  """
  result = pd.Series(index=df.index, dtype="float64")
  for _, idx in df.groupby("USER").groups.items():
    values = df.loc[idx, col]
    cap = values.quantile(q)
    cap = cap if pd.notna(cap) and cap > 0 else 1
    result.loc[idx] = 1.0 - (values / cap).clip(0, 1)
  return result


def weighted_nanmean(df: pd.DataFrame, columns: list[str], weights: list[float] | None = None) -> pd.Series:
  """ Compute a weighted row-wise mean while ignoring NaN components.

    Args:
      df: DataFrame containing component columns.
      columns: Component columns to average.
      weights: Optional component weights.

    Returns:
      pd.Series: Weighted row-wise means.
  """
  values = df[columns].to_numpy(dtype="float64")
  weights_arr = np.ones(len(columns), dtype="float64") if weights is None else np.array(weights, dtype="float64")
  mask = ~np.isnan(values)
  weighted_sum = np.nansum(values * weights_arr[np.newaxis, :], axis=1)
  weight_sum = np.sum(mask * weights_arr[np.newaxis, :], axis=1)
  return pd.Series(np.where(weight_sum > 0, weighted_sum / weight_sum, np.nan), index=df.index)


def compute_components(rolling_df: pd.DataFrame) -> pd.DataFrame:
  """ Compute BWCI v0.1 components and composite scores.

    Args:
      rolling_df: Rolling feature DataFrame.

    Returns:
      pd.DataFrame: DataFrame with component columns and BWCI scores.
  """
  out = rolling_df.copy()
  out["C_interaction_raw"] = (1.0 - out["idle_ratio"]).clip(0, 1)

  switch_stability = inverse_robust_scale_by_user(out, "switch_sum")
  foreground_stability = out["foreground_focus_share"].clip(0, 1)
  out["C_context_raw"] = pd.concat([switch_stability, foreground_stability], axis=1).mean(axis=1)

  out["C_input_raw"] = robust_percentile_by_user(out, "input_sum")

  enough_typing = out["keystroke_sum"] >= 5
  out["C_regularity_raw"] = np.nan
  out.loc[enough_typing & out["typing_cv"].notna(), "C_regularity_raw"] = (
    1.0 / (1.0 + out.loc[enough_typing & out["typing_cv"].notna(), "typing_cv"])
  ).clip(0, 1)

  out["C_correction_raw"] = np.nan
  correction_idx = enough_typing & out["erase_ratio"].notna()
  out.loc[correction_idx, "C_correction_raw"] = inverse_robust_scale_by_user(out.loc[correction_idx], "erase_ratio")

  raw_components = ["C_interaction_raw", "C_context_raw", "C_input_raw", "C_regularity_raw", "C_correction_raw"]
  for col in raw_components:
    out[col.replace("_raw", "_pu")] = robust_percentile_by_user(out, col)

  component_cols_raw = raw_components
  component_cols_pu = [col.replace("_raw", "_pu") for col in component_cols_raw]
  out["BWCI_abs"] = weighted_nanmean(out, component_cols_raw)
  out["BWCI_pu"] = weighted_nanmean(out, component_cols_pu)
  out["observed_component_count"] = out[component_cols_raw].notna().sum(axis=1)
  return out


bwci10 = compute_components(rolling10)
component_cols_raw = ["C_interaction_raw", "C_context_raw", "C_input_raw", "C_regularity_raw", "C_correction_raw"]
component_cols_pu = [col.replace("_raw", "_pu") for col in component_cols_raw]

bwci10_summary = pd.DataFrame(
  [
    {
      "rows": len(bwci10),
      "BWCI_pu_mean": bwci10["BWCI_pu"].mean(),
      "BWCI_pu_std": bwci10["BWCI_pu"].std(),
      "BWCI_pu_q25": bwci10["BWCI_pu"].quantile(0.25),
      "BWCI_pu_median": bwci10["BWCI_pu"].median(),
      "median_observed_components": bwci10["observed_component_count"].median(),
    }
  ]
)
bwci10_summary


,rows,BWCI_pu_mean,BWCI_pu_std,BWCI_pu_q25,BWCI_pu_median,median_observed_components
0,76149,0.4932,0.1313,0.3895,0.4969,5.0000


### Interpretacja BWCI v0.1

`BWCI_pu` jest normalizowane względem użytkownika, więc opisuje relatywną ciągłość zachowania danej osoby, a nie porównanie „kto pracuje lepiej”. To jest właściwsze dla BEHACOM, bo użytkownicy mają bardzo różne profile użycia komputera.

W notebooku modelowym nie będziemy traktować bieżącego `BWCI_pu` jako zwykłej cechy w głównym zestawie. Zostanie użyty jako osobny baseline autoregresyjny, bo jest metodologicznie ważnym punktem odniesienia.


## Sekcja 2: Target przyszłego niskiego BWCI

Tworzymy główny target `event_threshold_10`: wartość `1`, jeśli przyszły `BWCI_pu` za 10 obserwowanych rekordów spada poniżej dolnego kwartylu danego użytkownika. To jest proxy przyszłego wejścia w stan niskiej behawioralnej ciągłości pracy.

Równolegle zachowujemy `BWCI_future_10` jako target regresyjny dla późniejszego porównania modeli regresyjnych.


In [5]:
def add_future_targets(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
  """ Add future BWCI regression and classification targets.

    Args:
      df: DataFrame with USER, obs_idx, and BWCI_pu.
      horizon: Prediction horizon in observed rows.

    Returns:
      pd.DataFrame: DataFrame with future BWCI target and threshold event label.
  """
  out = df.sort_values(["USER", "obs_idx"]).copy()
  user_q25 = out.groupby("USER")["BWCI_pu"].transform(lambda values: values.quantile(0.25))
  future_col = f"BWCI_future_{horizon}"
  event_col = f"event_threshold_{horizon}"

  out[future_col] = out.groupby("USER")["BWCI_pu"].shift(-horizon)
  out[event_col] = (out[future_col] < user_q25).astype(float)
  out.loc[out[future_col].isna(), event_col] = np.nan
  out["BWCI_future_delta_10"] = out[future_col] - out["BWCI_pu"]
  return out


labeled = add_future_targets(bwci10, HORIZON)
target_balance = pd.DataFrame(
  [
    {
      "target": TARGET_COL,
      "valid_rows": int(labeled[TARGET_COL].notna().sum()),
      "positive_rows": int(labeled[TARGET_COL].sum()),
      "positive_pct": labeled[TARGET_COL].mean() * 100,
      "missing_rows": int(labeled[TARGET_COL].isna().sum()),
    }
  ]
)
target_balance


,target,valid_rows,positive_rows,positive_pct,missing_rows
0,event_threshold_10,76039,15637,20.5644,110


### Interpretacja targetu

Target ma dodatnią klasę na poziomie około jednej piątej obserwacji. To jest wystarczająco niezbalansowane, żeby w notebooku modelowym raportować AUC-PR obok AUC-ROC, ale nie tak skrajne, żeby pierwszy etap wymagał od razu specjalistycznych metod resamplingu.

AUC-ROC mierzy, jak dobrze model porządkuje przypadki pozytywne względem negatywnych w całym zakresie progów decyzyjnych. Przy niezbalansowanym targecie może jednak wyglądać dobrze nawet wtedy, gdy model słabo odzyskuje klasę pozytywną. AUC-PR skupia się na relacji precision-recall dla klasy pozytywnej, dlatego jest bardziej informatywne, gdy interesuje nas wykrywanie rzadszego zdarzenia ryzyka.

Brakujące targety na końcu serii każdego użytkownika są naturalne: dla ostatnich obserwacji nie mamy przyszłości `t + 10`. Te wiersze usuwamy dopiero po zbudowaniu cech.


## Sekcja 3: Cechy dostępne w czasie `t`

Budujemy dataset modelowy z cechami dostępnymi w czasie `t`. Rozróżniamy trzy zestawy cech:

- `baseline_autoregressive`: tylko bieżący `BWCI_pu`; to benchmark, który modele muszą pobić,
- `behavior_primary`: surowe rolling telemetry i cechy czasu, bez bieżącego BWCI i bez komponentów BWCI,
- `behavior_plus_bwci`: wersja diagnostyczna z dodanym bieżącym BWCI, żeby sprawdzić, ile daje informacja autoregresyjna.

Nie używamy jako cech kolumn przyszłych ani targetów (`future`, `event`).


In [6]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
  """ Add calendar-time features available at prediction time.

    Args:
      df: DataFrame containing datetime.

    Returns:
      pd.DataFrame: Copy with cyclical hour and day-of-week features.
  """
  out = df.copy()
  hour = out["datetime"].dt.hour + out["datetime"].dt.minute / 60
  dayofweek = out["datetime"].dt.dayofweek
  out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
  out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
  out["dow_sin"] = np.sin(2 * np.pi * dayofweek / 7)
  out["dow_cos"] = np.cos(2 * np.pi * dayofweek / 7)
  out["is_weekend"] = (dayofweek >= 5).astype(int)
  return out


def build_model_dataset(df: pd.DataFrame) -> pd.DataFrame:
  """ Build the modeling table and drop rows without the primary target.

    Args:
      df: Labeled BWCI DataFrame.

    Returns:
      pd.DataFrame: Modeling dataset with valid classification and regression targets.
  """
  out = add_time_features(df)
  out = out[out[TARGET_COL].notna() & out[REGRESSION_TARGET_COL].notna()].copy()
  out[TARGET_COL] = out[TARGET_COL].astype(int)
  return out.reset_index(drop=True)


model_dataset = build_model_dataset(labeled)

behavior_primary_features: list[str] = [
  "keystroke_sum",
  "mouse_sum",
  "word_sum",
  "erase_sum",
  "switch_sum",
  "idle_ratio",
  "press_press_mean",
  "press_press_std",
  "foreground_time_mean",
  "foreground_time_capped_sum",
  "active_apps_mean",
  "input_sum",
  "erase_ratio",
  "typing_cv",
  "foreground_focus_share",
  "observed_component_count",
  "hour_sin",
  "hour_cos",
  "dow_sin",
  "dow_cos",
  "is_weekend",
]
baseline_autoregressive_features: list[str] = ["BWCI_pu"]
behavior_plus_bwci_features: list[str] = behavior_primary_features + baseline_autoregressive_features
component_diagnostic_features: list[str] = component_cols_pu + ["observed_component_count", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]

feature_sets: dict[str, list[str]] = {
  "baseline_autoregressive": baseline_autoregressive_features,
  "behavior_primary": behavior_primary_features,
  "behavior_plus_bwci": behavior_plus_bwci_features,
  "component_diagnostic": component_diagnostic_features,
}

feature_set_summary = pd.DataFrame(
  [
    {
      "feature_set": name,
      "n_features": len(cols),
      "uses_current_bwci": "BWCI_pu" in cols,
      "uses_bwci_components": any(col.startswith("C_") for col in cols),
    }
    for name, cols in feature_sets.items()
  ]
)
feature_set_summary


,feature_set,n_features,uses_current_bwci,uses_bwci_components
0,baseline_autoregressive,1,True,False
1,behavior_primary,21,False,False
2,behavior_plus_bwci,22,True,False
3,component_diagnostic,10,False,True


### Interpretacja zestawów cech

Główny zestaw do pierwszych modeli to `behavior_primary`: ma `21` cech, nie zawiera bieżącego `BWCI_pu` i nie zawiera komponentów `C_*`. To ogranicza cyrkularność, bo model nie dostaje gotowego indeksu ani jego bezpośrednich składowych jako wejścia.

`baseline_autoregressive` ma tylko jedną cechę, `BWCI_pu`, i służy jako konieczny punkt odniesienia. Jeśli modele telemetryczne nie pobiją tego baseline’u, wynik trzeba interpretować jako silnie autoregresyjny, a nie jako sukces złożonego modelowania.

`behavior_plus_bwci` i `component_diagnostic` są wariantami diagnostycznymi, nie głównymi zestawami cech. Pomagają sprawdzić, ile daje bieżący indeks i jego komponenty, ale nie powinny być podstawą głównego wniosku o predykcji z surowej telemetrii.


In [7]:
def audit_feature_leakage(feature_sets_to_check: dict[str, list[str]]) -> pd.DataFrame:
  """ Check feature sets for obvious target leakage by column name.

    Args:
      feature_sets_to_check: Mapping of feature set name to feature columns.

    Returns:
      pd.DataFrame: Leakage audit table.
  """
  rows: list[dict[str, object]] = []
  forbidden_patterns = ["future", "event_"]
  for name, cols in feature_sets_to_check.items():
    forbidden_cols = [col for col in cols if any(pattern in col for pattern in forbidden_patterns)]
    missing_cols = [col for col in cols if col not in model_dataset.columns]
    rows.append(
      {
        "feature_set": name,
        "missing_cols": missing_cols,
        "forbidden_cols": forbidden_cols,
        "passes_name_check": not missing_cols and not forbidden_cols,
      }
    )
  return pd.DataFrame(rows)


leakage_audit = audit_feature_leakage(feature_sets)
leakage_audit


,feature_set,missing_cols,forbidden_cols,passes_name_check
0,baseline_autoregressive,[],[],True
1,behavior_primary,[],[],True
2,behavior_plus_bwci,[],[],True
3,component_diagnostic,[],[],True


In [8]:
missingness_rows: list[dict[str, object]] = []
for col in behavior_primary_features:
  missingness_rows.append(
    {
      "feature": col,
      "missing_pct": model_dataset[col].isna().mean() * 100,
      "median": model_dataset[col].median(),
    }
  )

missingness_df = pd.DataFrame(missingness_rows).sort_values("missing_pct", ascending=False)
missingness_df.head(12)


,feature,missing_pct,median
13,typing_cv,21.5416,1.7661
12,erase_ratio,19.1639,0.0405
2,word_sum,0.0000,10.0000
1,mouse_sum,0.0000,5707.0000
0,keystroke_sum,0.0000,88.0000
4,switch_sum,0.0000,1.0000
3,erase_sum,0.0000,3.0000
7,press_press_std,0.0000,605.2480
5,idle_ratio,0.0000,0.2000
8,foreground_time_mean,0.0000,56.6380


### Interpretacja jakości cech

Braki w `erase_ratio` i `typing_cv` są oczekiwane, bo te cechy mają sens tylko przy aktywności klawiaturowej. W notebooku 05 nie powinniśmy ich usuwać globalnie; pipeline modelowy powinien jawnie imputować wartości liczbowe, a dla modeli liniowych dodatkowo skalować cechy.

To jest też powód, dla którego `observed_component_count` zostaje jako cecha kontrolna: informuje, ile komponentów BWCI było obserwowalnych w danym oknie.


## Sekcja 4: Splity ewaluacyjne

Przygotowujemy dwa rodzaje podziału:

- split czasowy per użytkownik: `train / validation / test = 60% / 20% / 20%`,
- leave-one-user-out: lista foldów, w których jeden użytkownik jest testem.

Notebook 05 może zacząć od splitu czasowego, a leave-one-user-out wykorzystać jako mocniejszy test generalizacji między użytkownikami.


In [9]:
def assign_temporal_split(df: pd.DataFrame, train_frac: float = 0.60, validation_frac: float = 0.20) -> pd.Series:
  """ Assign per-user chronological train, validation, and test split labels.

    Args:
      df: Modeling dataset sorted by USER and obs_idx.
      train_frac: Fraction of each user's rows assigned to train.
      validation_frac: Fraction of each user's rows assigned to validation.

    Returns:
      pd.Series: Split labels aligned to df.
  """
  split = pd.Series(index=df.index, dtype="object")
  for _, idx in df.sort_values(["USER", "obs_idx"]).groupby("USER").groups.items():
    ordered_idx = list(idx)
    n_rows = len(ordered_idx)
    train_end = int(np.floor(n_rows * train_frac))
    validation_end = int(np.floor(n_rows * (train_frac + validation_frac)))
    split.loc[ordered_idx[:train_end]] = "train"
    split.loc[ordered_idx[train_end:validation_end]] = "validation"
    split.loc[ordered_idx[validation_end:]] = "test"
  return split


def summarize_split(df: pd.DataFrame, split_col: str) -> pd.DataFrame:
  """ Summarize row counts and positive target rate by split.

    Args:
      df: Modeling dataset with target and split label.
      split_col: Split column name.

    Returns:
      pd.DataFrame: Split summary table.
  """
  rows: list[dict[str, object]] = []
  for split_name, part in df.groupby(split_col):
    rows.append(
      {
        "split": split_name,
        "rows": len(part),
        "users": part["USER"].nunique(),
        "positive_pct": part[TARGET_COL].mean() * 100,
        "start": part["datetime"].min(),
        "end": part["datetime"].max(),
      }
    )
  return pd.DataFrame(rows)


model_dataset["temporal_split"] = assign_temporal_split(model_dataset)
temporal_split_summary = summarize_split(model_dataset, "temporal_split")
temporal_split_summary


,split,rows,users,positive_pct,start,end
0,test,15212,11,23.5801,2019-12-05 14:20:45.071,2020-01-14 09:43:38.352
1,train,45621,11,20.0281,2019-11-20 12:08:51.669,2020-01-10 21:06:01.996
2,validation,15206,11,19.1569,2019-12-03 15:22:00.529,2020-01-12 22:58:05.640


In [10]:
def summarize_leave_one_user_out(df: pd.DataFrame) -> pd.DataFrame:
  """ Summarize leave-one-user-out folds.

    Args:
      df: Modeling dataset with USER and target columns.

    Returns:
      pd.DataFrame: One row per test user fold.
  """
  rows: list[dict[str, object]] = []
  all_users = sorted(df["USER"].unique())
  for test_user in all_users:
    test = df[df["USER"] == test_user]
    train = df[df["USER"] != test_user]
    rows.append(
      {
        "test_user": int(test_user),
        "train_rows": len(train),
        "test_rows": len(test),
        "train_positive_pct": train[TARGET_COL].mean() * 100,
        "test_positive_pct": test[TARGET_COL].mean() * 100,
      }
    )
  return pd.DataFrame(rows)


logo_summary = summarize_leave_one_user_out(model_dataset)
logo_summary


,test_user,train_rows,test_rows,train_positive_pct,test_positive_pct
0,0,70728,5311,20.2310,25.0047
1,1,62589,13450,24.9804,0.0149
2,3,72903,3136,20.3764,24.9362
3,4,67929,8110,20.0312,25.0308
4,5,74317,1722,20.4583,25.1452
5,6,75057,982,20.5031,25.2546
6,7,72482,3557,20.3554,24.8243
7,8,71759,4280,20.3055,24.9065
8,9,70294,5745,20.2023,24.9956
9,10,63118,12921,19.6679,24.9439


### Interpretacja splitów

Split czasowy dzieli dataset na `45,621` wierszy train, `15,206` validation i `15,212` test. Positive rate wynosi odpowiednio `20.03%`, `19.16%` i `23.58%`, więc test ma nieco więcej klasy pozytywnej niż validation i train. To trzeba uwzględnić przy interpretacji AUC-PR, bo jego baseline zależy od częstości klasy pozytywnej.

Leave-one-user-out pokazuje istotny problem User 1: jego fold testowy ma tylko `0.0149%` pozytywnych przypadków, podczas gdy większość pozostałych użytkowników ma około `25%`. To już w notebooku 04 sygnalizuje, że target progowy może być niestabilny per user.

Decyzja dla notebooka 05: zaczynamy od temporal split, ale metryki per user i LOGO muszą być obowiązkową kontrolą. Sam wynik globalny nie wystarczy.


## Sekcja 5: Podgląd datasetu i zapis artefaktu

Zapisujemy lokalny parquet w `outputs/`, żeby notebook 05 mógł zacząć od gotowego datasetu modelowego.

In [11]:
model_dataset_summary = pd.DataFrame(
  [
    {
      "rows": len(model_dataset),
      "users": model_dataset["USER"].nunique(),
      "columns": len(model_dataset.columns),
      "target": TARGET_COL,
      "positive_pct": model_dataset[TARGET_COL].mean() * 100,
      "regression_target": REGRESSION_TARGET_COL,
      "bwci_future_mean": model_dataset[REGRESSION_TARGET_COL].mean(),
      "bwci_future_std": model_dataset[REGRESSION_TARGET_COL].std(),
    }
  ]
)
model_dataset_summary


,rows,users,columns,target,positive_pct,regression_target,bwci_future_mean,bwci_future_std
0,76039,11,41,event_threshold_10,20.5644,BWCI_future_10,0.4932,0.1314


In [12]:
preview_cols = [
  "USER",
  "obs_idx",
  "datetime",
  "BWCI_pu",
  REGRESSION_TARGET_COL,
  TARGET_COL,
  "temporal_split",
] + behavior_primary_features[:8]

model_dataset[preview_cols].head(10)


,USER,obs_idx,datetime,BWCI_pu,BWCI_future_10,event_threshold_10,temporal_split,keystroke_sum,mouse_sum,word_sum,erase_sum,switch_sum,idle_ratio,press_press_mean,press_press_std
0,0,4,2019-11-20 13:48:20.304,0.4182,0.3955,1,train,174.0000,4425.0000,13.0000,11.0000,8.0000,0.0000,1187.0640,2544.5460
1,0,5,2019-11-20 13:49:22.392,0.4392,0.3859,1,train,194.0000,5597.0000,15.0000,12.0000,10.0000,0.0000,1035.9300,2152.7083
2,0,6,2019-11-20 13:50:23.466,0.4569,0.3590,1,train,202.0000,6864.0000,16.0000,13.0000,10.0000,0.0000,1461.6957,2959.3586
3,0,7,2019-11-20 13:51:26.309,0.5626,0.2891,1,train,205.0000,8417.0000,16.0000,13.0000,14.0000,0.0000,4286.0463,5124.5050
4,0,8,2019-11-20 13:52:31.149,0.5617,0.2853,1,train,205.0000,8532.0000,16.0000,13.0000,16.0000,0.0000,3809.8189,4555.1156
5,0,9,2019-11-20 13:53:34.810,0.5189,0.2853,1,train,205.0000,8532.0000,16.0000,13.0000,16.0000,0.1000,3428.8370,4099.6040
6,0,10,2019-11-20 13:54:35.751,0.4995,0.3394,1,train,191.0000,8362.0000,14.0000,11.0000,14.0000,0.2000,3408.7680,4083.0570
7,0,11,2019-11-20 13:55:41.168,0.4778,0.3907,1,train,154.0000,6612.0000,12.0000,8.0000,12.0000,0.3000,3326.1990,3898.7980
8,0,12,2019-11-20 13:56:45.662,0.4528,0.3526,1,train,130.0000,5993.0000,9.0000,8.0000,10.0000,0.4000,3086.2120,3380.6430
9,0,13,2019-11-20 13:57:49.386,0.4135,0.3919,1,train,102.0000,5414.0000,8.0000,8.0000,11.0000,0.5000,2877.4340,2953.3690


In [13]:
metadata: dict[str, object] = {
  "dataset_version": "v0.1",
  "primary_index": "BWCI_pu",
  "window": WINDOW,
  "horizon": HORIZON,
  "classification_target": TARGET_COL,
  "regression_target": REGRESSION_TARGET_COL,
  "feature_sets": feature_sets,
  "temporal_split_summary": temporal_split_summary.astype(str).to_dict(orient="records"),
  "leave_one_user_out_summary": logo_summary.round(4).to_dict(orient="records"),
}

model_dataset.to_parquet(OUTPUT_DATASET_PATH, index=False)
OUTPUT_METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))

print(f"Saved dataset: {OUTPUT_DATASET_PATH} ({len(model_dataset):,} rows)")
print(f"Saved metadata: {OUTPUT_METADATA_PATH}")


Saved dataset: /home/sefni/git/bwci-thesis/outputs/model_dataset_event_threshold_10_v0_1.parquet (76,039 rows)
Saved metadata: /home/sefni/git/bwci-thesis/outputs/model_dataset_event_threshold_10_v0_1_metadata.json


## Sekcja 6: Decyzje dla notebooka 05

Przyjmujemy na start:

1. Główne zadanie klasyfikacyjne: `event_threshold_10`.
2. Główne cechy ML: `behavior_primary`.
3. Minimalny baseline: `baseline_autoregressive`, czyli bieżący `BWCI_pu`.
4. Ewaluacja: split czasowy jako pierwszy wynik, leave-one-user-out jako test generalizacji.
5. Metryki klasyfikacyjne: AUC-ROC, AUC-PR, balanced accuracy, recall/precision dla klasy pozytywnej.
6. Dla regresji pomocniczej: MAE, RMSE i korelacja Spearmana/Pearsona dla `BWCI_future_10`.

W notebooku 05 nie należy zaczynać od najbardziej złożonych modeli. Najpierw DummyClassifier / DummyRegressor i baseline autoregresyjny, dopiero potem Logistic Regression, Random Forest i ewentualnie gradient boosting.
